# R1 — Tools as Unity Catalog functions

Creates the four governed tools in `claims.genai` for the Saffron Shield adjuster assistant: `lookup_claim`, `plan_at`, `search_policy` and `compute_payable`. Claim facts come from the gold table exactly as stored, the plan comes from the policy version valid on the requested date, policy answers come from hybrid keyword-plus-vector search over the Paper 4 chunk index, and the payout figure comes from one deterministic function that never calls a model.

In [0]:
# Config — the single place the catalog and models are named for this notebook
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"

spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  chat_model={CHAT_MODEL}  embed_model={EMBED_MODEL}")

In [0]:
# Tool 1 — claim facts (no payout decision)
# Tool 2 — the plan a member held on a given date (policy version valid on that date)
spark.sql("""
CREATE OR REPLACE FUNCTION claims.genai.lookup_claim(claim_id STRING)
RETURNS TABLE (
  claim_id STRING, member_id STRING, hospital_id STRING, status STRING, claim_type STRING,
  admission_date DATE, discharge_date DATE, billed_amount DOUBLE, approved_amount DOUBLE,
  rejection_code STRING, is_pre_existing BOOLEAN, plan_code_at_admission STRING, plan_family STRING,
  room_rent_per_day INT, room_rent_cap_per_day INT, room_rent_over_cap BOOLEAN,
  policy_start_date DATE, policy_tenure_months INT, days_since_policy_start INT,
  is_network_at_admission BOOLEAN
)
COMMENT 'Claim facts for one claim, exactly as stored in claims.gold.claims_enriched (no recomputation), plus days_since_policy_start = days from the first policy start date to admission. Does not decide a payout.'
RETURN
SELECT
  c.claim_id, c.member_id, c.hospital_id, c.status, c.claim_type,
  to_date(c.admission_date) AS admission_date,
  to_date(c.discharge_date) AS discharge_date,
  c.billed_amount, c.approved_amount, c.rejection_code, c.is_pre_existing,
  c.plan_code_at_admission, c.plan_family,
  CAST(c.room_rent_per_day AS INT) AS room_rent_per_day,
  CAST(c.room_rent_cap_per_day AS INT) AS room_rent_cap_per_day,
  c.room_rent_over_cap, c.policy_start_date, c.policy_tenure_months,
  CAST(datediff(to_date(c.admission_date), c.policy_start_date) AS INT) AS days_since_policy_start,
  c.is_network_at_admission
FROM claims.gold.claims_enriched c
WHERE c.claim_id = lookup_claim.claim_id
""")

spark.sql("""
CREATE OR REPLACE FUNCTION claims.genai.plan_at(member_id STRING, as_of_date DATE)
RETURNS TABLE (
  member_id STRING, policy_id STRING, plan_code STRING, plan_family STRING,
  room_rent_cap_per_day INT, copay_pct INT, ped_waiting_months INT, initial_waiting_days INT,
  valid_from DATE, valid_to DATE
)
COMMENT 'The plan terms the member held on as_of_date: the policy version from claims.silver.policy_history whose valid_from is on or before the date and whose valid_to is after it (empty valid_to = still current), joined to plan terms in claims.raw.plans. Comparison is by date.'
RETURN
SELECT
  ph.member_id, ph.policy_id, ph.plan_code,
  pl.plan_family,
  CAST(pl.room_rent_cap_per_day AS INT) AS room_rent_cap_per_day,
  CAST(pl.copay_pct AS INT) AS copay_pct,
  CAST(pl.ped_waiting_months AS INT) AS ped_waiting_months,
  CAST(pl.initial_waiting_days AS INT) AS initial_waiting_days,
  ph.__START_AT AS valid_from,
  ph.__END_AT AS valid_to
FROM claims.silver.policy_history ph
JOIN claims.raw.plans pl
  ON pl.plan_code = ph.plan_code
WHERE ph.member_id = plan_at.member_id
  AND ph.__START_AT <= as_of_date
  AND (ph.__END_AT IS NULL OR ph.__END_AT > as_of_date)
""")

print("lookup_claim + plan_at created")

In [0]:
# Tool 3 — hybrid keyword + vector search over the policy binder (top 5, best first)
# Vector recall from the Paper 4 index, re-ranked with a keyword-overlap score.
spark.sql("""
CREATE OR REPLACE FUNCTION claims.genai.search_policy(query STRING)
RETURNS TABLE (
  chunk_id STRING, doc_name STRING, heading STRING, text STRING
)
COMMENT 'Hybrid keyword-plus-vector search of the Saffron Shield policy binder (index claims.genai.policy_chunks_idx). Returns at most 5 chunks, best first. chunk_id is the bare chunk id (docname#anchor).'
RETURN
WITH vs AS (
  SELECT chunk_id, search_score
  FROM vector_search(
    index => 'claims.genai.policy_chunks_idx',
    query => search_policy.query,
    num_results => 20
  )
), kw AS (
  SELECT
    pc.chunk_id, pc.doc_name, pc.heading, pc.text,
    size(
      filter(
        split(lower(search_policy.query), '[^a-z0-9]+'),
        t -> length(t) >= 3 AND (contains(lower(pc.heading), t) OR contains(lower(pc.text), t))
      )
    ) AS keyword_hits
  FROM claims.genai.policy_chunks pc
), scored AS (
  SELECT
    kw.chunk_id, kw.doc_name, kw.heading, kw.text,
    0.7 * (vs.search_score / max(vs.search_score) OVER ()) + 0.3 * (kw.keyword_hits / greatest(1, size(filter(split(lower(search_policy.query), '[^a-z0-9]+'), t -> length(t) >= 3)))) AS hybrid_score
  FROM vs JOIN kw ON kw.chunk_id = vs.chunk_id
)
SELECT chunk_id, doc_name, heading, text
FROM scored
ORDER BY hybrid_score DESC
LIMIT 5
""")

print("search_policy created")

In [0]:
# Tool 4 — deterministic payout decision (NEVER calls a model).
# Rules straight from the assignment Definitions: waiting rules, room rent deduction, co-pay, reason code, payable amount.
# Plan terms are pulled with max() over the (unique) plan_code row because a correlated subquery in a scalar UC function must aggregate.
spark.sql("""
CREATE OR REPLACE FUNCTION claims.genai.compute_payable(
  plan_code STRING,
  is_pre_existing BOOLEAN,
  policy_tenure_months INT,
  days_since_policy_start INT,
  is_network_at_admission BOOLEAN,
  room_rent_per_day INT,
  billed_amount DOUBLE
)
RETURNS STRING
COMMENT 'Decides a claim by the Definitions and returns a JSON string with the payout: plan_code, room_rent_cap_per_day, room_rent_over_cap, waiting_period_ok, payable, network_at_admission, room_rent_deduction_pct (2 decimals), copay_pct, reason_code (INITIAL_WAITING / PED_WAITING / PAYABLE_WITH_DEDUCTION / PAYABLE_IN_FULL) and payable_amount (2 decimals). Deterministic only - never calls a model.'
RETURN
SELECT to_json(named_struct(
  'plan_code', compute_payable.plan_code,
  'room_rent_cap_per_day', pm.cap,
  'room_rent_over_cap', room_rent_per_day > pm.cap,
  'waiting_period_ok', days_since_policy_start >= pm.init_d
      AND (NOT is_pre_existing OR policy_tenure_months >= pm.ped_m),
  'payable', days_since_policy_start >= pm.init_d
      AND (NOT is_pre_existing OR policy_tenure_months >= pm.ped_m),
  'network_at_admission', is_network_at_admission,
  'room_rent_deduction_pct', CASE
      WHEN room_rent_per_day <= pm.cap THEN 0.0
      ELSE CAST(round((room_rent_per_day - pm.cap) * 100.0 / room_rent_per_day, 2) AS DOUBLE)
    END,
  'copay_pct', pm.copay_base + CASE WHEN NOT is_network_at_admission THEN 20 ELSE 0 END,
  'reason_code', CASE
      WHEN NOT (days_since_policy_start >= pm.init_d) THEN 'INITIAL_WAITING'
      WHEN NOT (NOT is_pre_existing OR policy_tenure_months >= pm.ped_m) THEN 'PED_WAITING'
      WHEN (CASE WHEN room_rent_per_day <= pm.cap THEN 0.0
                ELSE CAST(round((room_rent_per_day - pm.cap) * 100.0 / room_rent_per_day, 2) AS DOUBLE) END) > 0
        OR (pm.copay_base + CASE WHEN NOT is_network_at_admission THEN 20 ELSE 0 END) > 0
      THEN 'PAYABLE_WITH_DEDUCTION'
      ELSE 'PAYABLE_IN_FULL'
    END,
  'payable_amount', CASE
      WHEN days_since_policy_start >= pm.init_d
        AND (NOT is_pre_existing OR policy_tenure_months >= pm.ped_m)
      THEN CAST(round(
             (billed_amount * (1 - (CASE WHEN room_rent_per_day <= pm.cap THEN 0.0
                ELSE CAST(round((room_rent_per_day - pm.cap) * 100.0 / room_rent_per_day, 2) AS DOUBLE) END) / 100))
             * (1 - (pm.copay_base + CASE WHEN NOT is_network_at_admission THEN 20 ELSE 0 END) / 100), 2) AS DOUBLE)
      ELSE CAST(0.0 AS DOUBLE)
    END
))
FROM (
  SELECT max(named_struct(
    'cap', CAST(room_rent_cap_per_day AS INT),
    'copay_base', CAST(copay_pct AS INT),
    'ped_m', CAST(ped_waiting_months AS INT),
    'init_d', CAST(initial_waiting_days AS INT)
  )) AS pm
  FROM claims.raw.plans
  WHERE plan_code = compute_payable.plan_code
)
""")

print("compute_payable created")

In [0]:
# Verify lookup_claim (claim facts as stored) and plan_at (plan valid on a date)
display(spark.sql("SELECT * FROM claims.genai.lookup_claim('CLM-20003')"))
display(spark.sql("SELECT * FROM claims.genai.plan_at('MEM-0493', DATE'2026-01-20')"))
display(spark.sql("SELECT * FROM claims.genai.plan_at('MEM-0493', DATE'2026-02-01')"))

In [0]:
# Verify search_policy: top-5, best first, bare chunk ids
# (policy probe question reused verbatim in R2)
display(spark.sql("""
SELECT chunk_id, doc_name, heading
FROM claims.genai.search_policy('What is the pre-existing disease (PED) waiting period for a corporate group plan?')
"""))

display(spark.sql("""
SELECT chunk_id, doc_name, heading
FROM claims.genai.search_policy('room rent capping and proportionate deduction')
"""))

In [0]:
# Verify compute_payable on the 20 Paper 4 probe claims, against the independent verifier answer key
import json

payouts = spark.sql("""
SELECT p.claim_id,
       claims.genai.compute_payable(
         c.plan_code_at_admission, c.is_pre_existing, c.policy_tenure_months,
         CAST(datediff(to_date(c.admission_date), c.policy_start_date) AS INT),
         c.is_network_at_admission, CAST(c.room_rent_per_day AS INT), c.billed_amount
       ) AS payout
FROM claims.genai.adjudication_probes p
JOIN claims.gold.claims_enriched c ON c.claim_id = p.claim_id
ORDER BY p.claim_id
""")
display(payouts)

key = {r.claim_id: r for r in spark.sql("SELECT * FROM claims.genai.adjudication_verifier").collect()}

mismatches = 0
for r in payouts.collect():
    j = json.loads(r.payout)
    k = key[r.claim_id]
    ok = (
        j["payable"] == k.payable
        and j["waiting_period_ok"] == k.waiting_period_ok
        and j["network_at_admission"] == k.network_at_admission
        and j["copay_pct"] == k.copay_pct
        and j["reason_code"] == k.reason_code
        and abs(j["room_rent_deduction_pct"] - (k.room_rent_deduction_pct or 0.0)) < 1.0
    )
    if not ok:
        mismatches += 1
        print(f"MISMATCH {r.claim_id}: tool={j}  key=payable={k.payable} waiting={k.waiting_period_ok} net={k.network_at_admission} ded={k.room_rent_deduction_pct} copay={k.copay_pct} reason={k.reason_code}")

print(f"{payouts.count()} probe claims checked, {mismatches} mismatches vs the Paper 4 verifier key")